# 🧠 SDNC Pipeline — Distillation Teacher → Student sur A100

Pipeline automatisé complet :
```
Colab A100 (Qwen3.5-35B-A3B BF16)
     │ train + benchmark
     ▼
Distillation sommeil 35B → 4B (via ProjectionBridge)
     │
     ▼
Benchmark 4B distillé
     │
     ▼
Google Cloud Storage → GitHub → PC local RX 7800 XT
```

**Variables à remplir** dans la cellule Setup ci-dessous.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 1 — Setup & Auth
# ═══════════════════════════════════════════════════════════════

# --- Variables configurables (À REMPLIR) ---
GITHUB_REPO = ""          # ex: "https://github.com/user/sdnc"
GITHUB_TOKEN = ""         # depuis Colab Secrets : userdata.get('GITHUB_TOKEN')
GCS_BUCKET = "sdnc-models"
GCS_PROJECT = ""          # ex: "my-gcp-project"
N_CYCLES = 10
N_TRAIN_STEPS_PER_CYCLE = 500

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Auth GCS ---
from google.colab import auth
auth.authenticate_user()

# --- Secrets Colab (si disponibles) ---
try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

import os
if GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
if GCS_PROJECT:
    os.environ['GOOGLE_CLOUD_PROJECT'] = GCS_PROJECT

# --- Installation dépendances ---
!pip install -q transformers torch accelerate bitsandbytes \
    google-cloud-storage gitpython snntorch ncps tqdm

# --- Clone repo SDNC ---
if GITHUB_REPO:
    repo_url = GITHUB_REPO
    if GITHUB_TOKEN:
        repo_url = repo_url.replace("https://", f"https://{GITHUB_TOKEN}@")
    !git clone {repo_url} /content/sdnc 2>/dev/null || (cd /content/sdnc && git pull)
    os.chdir('/content/sdnc')
else:
    print("⚠ GITHUB_REPO non configuré — assurez-vous que le code est disponible.")

# --- Vérification GPU ---
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU : {gpu} | VRAM : {vram:.0f} GB")
else:
    raise RuntimeError("❌ Pas de GPU — sélectionnez un runtime A100.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 2 — Audit modèles
# ═══════════════════════════════════════════════════════════════

from transformers import AutoConfig

# --- Teacher : Qwen3.5-35B-A3B ---
print("Audit Teacher : Qwen/Qwen3.5-35B-A3B")
try:
    teacher_cfg = AutoConfig.from_pretrained(
        "Qwen/Qwen3.5-35B-A3B", trust_remote_code=True
    )
    teacher_hidden = teacher_cfg.hidden_size
    teacher_layers = teacher_cfg.num_hidden_layers
    print(f"  hidden_size   : {teacher_hidden}")
    print(f"  num_layers    : {teacher_layers}")
    assert teacher_hidden > 0, "hidden_size invalide"
except Exception as e:
    raise RuntimeError(f"❌ Audit teacher ÉCHOUÉ : {e}")

# --- Student : Qwen3.5-4B ---
print("\nAudit Student : Qwen/Qwen3.5-4B")
try:
    student_cfg = AutoConfig.from_pretrained(
        "Qwen/Qwen3.5-4B", trust_remote_code=True
    )
    student_hidden = student_cfg.hidden_size
    student_layers = student_cfg.num_hidden_layers
    print(f"  hidden_size   : {student_hidden}")
    print(f"  num_layers    : {student_layers}")
    assert student_hidden == 2560, f"Attendu 2560, trouvé {student_hidden}"
except Exception as e:
    raise RuntimeError(f"❌ Audit student ÉCHOUÉ : {e}")

# --- VRAM ---
import torch
if torch.cuda.is_available():
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_free = (torch.cuda.get_device_properties(0).total_memory
                 - torch.cuda.memory_allocated(0)) / 1e9
    print(f"\nVRAM disponible : {vram_free:.1f} / {vram_total:.0f} GB")

    # Estimation VRAM nécessaire
    teacher_vram_est = teacher_hidden * teacher_layers * 4 * 2 / 1e9  # rough BF16
    student_vram_est = student_hidden * student_layers * 4 * 2 / 1e9
    print(f"  Teacher ~{teacher_vram_est:.0f} GB (BF16) | Student ~{student_vram_est:.0f} GB (BF16)")

print("\n✓ Audit PASS")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 3 — Initialisation pipeline
# ═══════════════════════════════════════════════════════════════

from pipeline.dual_model_config import DualModelConfig
from pipeline.full_loop import SDNCPipeline

# Configuration avec les valeurs auditées
config = DualModelConfig(
    teacher_model="Qwen/Qwen3.5-35B-A3B",
    teacher_hidden=teacher_hidden,  # depuis l'audit
    student_model="Qwen/Qwen3.5-4B",
    student_hidden=student_hidden,  # depuis l'audit
    gcs_bucket=GCS_BUCKET,
    github_repo=GITHUB_REPO,
    github_branch="main",
)

# Initialiser le pipeline
pipeline = SDNCPipeline(config)

# Audit complet (charge teacher + student + test dimensions)
audit_results = pipeline.audit()

# Reprendre depuis un checkpoint si disponible
pipeline.resume()

# Afficher l'état
print(pipeline.status())

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 4 — Lancement boucle complète
# ═══════════════════════════════════════════════════════════════

try:
    pipeline.run_full_cycle(n_train_steps=N_TRAIN_STEPS_PER_CYCLE)
except KeyboardInterrupt:
    pipeline.emergency_save()
    print("\n✓ Interrompu proprement — checkpoint sauvegardé")
except Exception as e:
    pipeline.emergency_save()
    print(f"\n❌ Erreur : {e}")
    raise

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 5 — Monitoring (exécutable en parallèle)
# ═══════════════════════════════════════════════════════════════

import time
from IPython.display import clear_output

try:
    while True:
        clear_output(wait=True)
        print(pipeline.status())
        print(f"\n  Heure : {time.strftime('%H:%M:%S')}")

        if torch.cuda.is_available():
            vram_used = torch.cuda.memory_allocated(0) / 1e9
            vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
            pct = vram_used / vram_total * 100
            bar_len = 30
            filled = int(bar_len * pct / 100)
            bar = "█" * filled + "░" * (bar_len - filled)
            print(f"  VRAM [{bar}] {pct:.0f}%")

        time.sleep(60)
except KeyboardInterrupt:
    print("Monitoring arrêté.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 6 — Test local_runner (simulation PC)
# ═══════════════════════════════════════════════════════════════
# Simule ce que fera le PC local après git pull + sync GCS.

from pipeline.local_runner import LocalRunner

runner = LocalRunner(config=config)

# Charger le student depuis le dernier checkpoint
model = runner.load_student()

if model is not None:
    # Test de génération post-distillation
    result = model.forward("Explique le predictive coding dans SDNC", learn=False)
    print(f"Réponse : {result['response'][:500]}")
    print(f"\nMétriques :")
    print(f"  Erreur moyenne   : {result['mean_error']:.4f}")
    print(f"  Conflict score   : {result['conflict_score']:.4f}")
    print(f"  PC errors        : {result['pc_errors']}")
    print(f"  Scheduler phase  : {result['scheduler_phase']}")
    print(f"  Mémoires stockées: {result['memories_stored']}")
else:
    print("Aucun modèle chargé — lancez d'abord le pipeline (cellule 4).")